In [1]:
import pytesseract
print(pytesseract.get_tesseract_version())

4.1.1


In [2]:
import os

from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector

# Backend interactivo (mejor opción en Jupyter moderno)
%matplotlib notebook

print("Imports OK ✅")

Imports OK ✅


In [3]:
def load_pdf_as_images(pdf_path: str, dpi: int = 200):
    """
    Carga un PDF y lo convierte en una lista de imágenes PIL (una por página).
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"No se encontró el archivo: {pdf_path}")
    
    print(f"Cargando PDF desde: {pdf_path}")
    pages = convert_from_path(pdf_path, dpi=dpi)
    print(f"PDF cargado: {len(pages)} páginas encontradas ✅")
    return pages

print("Función load_pdf_as_images lista ✅")

Función load_pdf_as_images lista ✅


In [4]:
pdf_path = "Santander.pdf"  # <<--- cámbialo

pages = load_pdf_as_images(pdf_path, dpi=200)

# Mostrar SOLO para probar que se ve algo
fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(pages[0])
ax.set_title("Prueba: Página 0")
ax.axis("off")
plt.show()

Cargando PDF desde: Santander.pdf
PDF cargado: 4 páginas encontradas ✅


<IPython.core.display.Javascript object>

In [5]:
import tkinter as tk
from PIL import ImageTk
import pytesseract

def browse_and_select_region_tk(pages, start_page: int = 0, max_width: int = 1000, max_height: int = 800):
    """
    Abre una ventana Tkinter para navegar por las páginas de 'pages'
    con botones Anterior/Siguiente y seleccionar recuadros con el mouse.

    Cada vez que seleccionas un recuadro:
      - Imprime en consola:
          - número de página
          - bbox en píxeles (imagen original)
          - texto OCR dentro del recuadro
          - texto OCR de toda la página (truncado)
    La ventana NO se cierra, puedes seguir cambiando de página y seleccionando.
    """
    if not pages:
        raise ValueError("La lista de páginas está vacía.")
    if start_page < 0 or start_page >= len(pages):
        raise IndexError(f"start_page fuera de rango (0–{len(pages)-1})")

    state = {
        "pages": pages,
        "page_index": start_page,
        "max_width": max_width,
        "max_height": max_height,
        "canvas": None,
        "root": None,
        "tk_image": None,
        "rect_id": None,
        "x0": None,
        "y0": None,
        "scale": 1.0,
    }

    root = tk.Tk()
    root.title("Navegador PDF - selecciona recuadro")
    state["root"] = root

    # Canvas donde se dibuja la página
    canvas = tk.Canvas(root, width=max_width, height=max_height, bg="gray")
    canvas.grid(row=0, column=0, columnspan=3)
    state["canvas"] = canvas

    def show_page():
        """Muestra la página actual en el canvas, redimensionada si es necesario."""
        pil_image = state["pages"][state["page_index"]]
        orig_w, orig_h = pil_image.size

        s = min(state["max_width"] / orig_w, state["max_height"] / orig_h, 1.0)
        disp_w = int(orig_w * s)
        disp_h = int(orig_h * s)

        disp_image = pil_image.resize((disp_w, disp_h), Image.LANCZOS)

        tk_image = ImageTk.PhotoImage(disp_image)
        state["tk_image"] = tk_image  # evitar GC

        canvas.config(width=disp_w, height=disp_h)
        canvas.delete("all")
        canvas.create_image(0, 0, anchor="nw", image=tk_image)

        state["scale"] = s
        state["rect_id"] = None
        state["x0"], state["y0"] = None, None

        root.title(f"Página {state['page_index']} de {len(state['pages']) - 1} - dibuja un recuadro")

    def on_prev():
        if state["page_index"] > 0:
            state["page_index"] -= 1
            show_page()

    def on_next():
        if state["page_index"] < len(state["pages"]) - 1:
            state["page_index"] += 1
            show_page()

    def on_button_press(event):
        state["x0"], state["y0"] = event.x, event.y
        if state["rect_id"] is not None:
            canvas.delete(state["rect_id"])
        state["rect_id"] = canvas.create_rectangle(
            state["x0"], state["y0"], event.x, event.y,
            outline="red", width=2
        )

    def on_move(event):
        if state["rect_id"] is not None:
            canvas.coords(state["rect_id"], state["x0"], state["y0"], event.x, event.y)

    def on_button_release(event):
        x0, y0 = state["x0"], state["y0"]
        x1, y1 = event.x, event.y

        if x0 is None or y0 is None:
            print("Selección inválida.")
            return

        left_disp, right_disp = sorted([x0, x1])
        top_disp, bottom_disp = sorted([y0, y1])

        s = state["scale"]
        # Convertir a coordenadas de la imagen original
        left = int(left_disp / s)
        right = int(right_disp / s)
        top = int(top_disp / s)
        bottom = int(bottom_disp / s)

        if right <= left or bottom <= top:
            print("Recuadro demasiado pequeño o inválido.")
            return

        bbox = (left, top, right, bottom)
        pil_image = state["pages"][state["page_index"]]
        cropped = pil_image.crop(bbox)

        print("\n==============================")
        print(f"📄 Página seleccionada: {state['page_index']}")
        print(f"📌 BBOX (px, original): {bbox}")
        print("==============================\n")

        # OCR de la región
        text_region = pytesseract.image_to_string(cropped, lang="spa")
        print("🟦 Texto dentro del recuadro:\n")
        print(text_region.strip() or "[Vacío / no se reconoció texto]")

        # OCR de la página completa
        text_page = pytesseract.image_to_string(pil_image, lang="spa")
        print("\n------------------------------")
        print("📄 Texto completo de la página (primeros 2000 caracteres):\n")
        print((text_page.strip() or "[Vacío / no se reconoció texto]")[:2000])
        print("\n==============================\n")

    # Botones de navegación
    btn_prev = tk.Button(root, text="⟵ Anterior", command=on_prev)
    btn_prev.grid(row=1, column=0, sticky="ew")

    btn_close = tk.Button(root, text="Cerrar", command=root.destroy)
    btn_close.grid(row=1, column=1, sticky="ew")

    btn_next = tk.Button(root, text="Siguiente ⟶", command=on_next)
    btn_next.grid(row=1, column=2, sticky="ew")

    # Eventos de ratón
    canvas.bind("<ButtonPress-1>", on_button_press)
    canvas.bind("<B1-Motion>", on_move)
    canvas.bind("<ButtonRelease-1>", on_button_release)

    show_page()
    print("Ventana abierta: usa Anterior/Siguiente para cambiar de página y dibuja un recuadro.")
    root.mainloop()

In [6]:
import tkinter as tk
from PIL import ImageTk

def select_region_tk(pages, page_index: int = 0, max_width: int = 1000, max_height: int = 800):
    """
    Abre una ventana Tkinter con la página indicada,
    permite seleccionar un recuadro con el mouse y,
    al soltar, hace OCR de la región y de la página completa.
    
    Imprime en consola:
      - Número de página
      - Texto dentro del recuadro
      - Texto de la página completa
    """
    if page_index < 0 or page_index >= len(pages):
        raise IndexError(f"page_index fuera de rango (0–{len(pages)-1})")
    
    pil_image = pages[page_index]
    orig_w, orig_h = pil_image.size
    
    # Escalar la imagen para que quepa en la ventana sin perder la relación de aspecto
    scale = min(max_width / orig_w, max_height / orig_h, 1.0)
    disp_w = int(orig_w * scale)
    disp_h = int(orig_h * scale)
    disp_image = pil_image.resize((disp_w, disp_h), Image.LANCZOS)
    
    # Crear ventana
    root = tk.Tk()
    root.title(f"Página {page_index} - dibuja un recuadro")
    
    canvas = tk.Canvas(root, width=disp_w, height=disp_h)
    canvas.pack()
    
    tk_image = ImageTk.PhotoImage(disp_image)
    # Guardar referencia para que no se la lleve el GC
    canvas.image = tk_image
    canvas.create_image(0, 0, anchor="nw", image=tk_image)
    
    state = {
        "x0": None,
        "y0": None,
        "rect_id": None,
        "scale": scale,
        "page_index": page_index,
        "pil_image": pil_image,
    }
    
    def on_button_press(event):
        state["x0"], state["y0"] = event.x, event.y
        if state["rect_id"] is not None:
            canvas.delete(state["rect_id"])
        state["rect_id"] = canvas.create_rectangle(
            state["x0"], state["y0"], event.x, event.y,
            outline="red", width=2
        )
    
    def on_move(event):
        if state["rect_id"] is not None:
            canvas.coords(state["rect_id"], state["x0"], state["y0"], event.x, event.y)
    
    def on_button_release(event):
        x0, y0 = state["x0"], state["y0"]
        x1, y1 = event.x, event.y
        
        if x0 is None or y0 is None:
            print("Selección inválida.")
            return
        
        # Coordenadas en la imagen mostrada (escalada)
        left_disp   = min(x0, x1)
        right_disp  = max(x0, x1)
        top_disp    = min(y0, y1)
        bottom_disp = max(y0, y1)
        
        # Convertir a coordenadas de la imagen original
        s = state["scale"]
        left   = int(left_disp / s)
        right  = int(right_disp / s)
        top    = int(top_disp / s)
        bottom = int(bottom_disp / s)
        
        # Evitar recuadros degenerados
        if right <= left or bottom <= top:
            print("Recuadro demasiado pequeño o inválido.")
            root.destroy()
            return
        
        bbox = (left, top, right, bottom)
        cropped = state["pil_image"].crop(bbox)
        
        print("\n==============================")
        print(f"📄 Página seleccionada: {state['page_index']}")
        print(f"📌 BBOX (px, original): {bbox}")
        print("==============================\n")
        
        # OCR de la región
        text_region = pytesseract.image_to_string(cropped, lang="spa")
        print("🟦 Texto dentro del recuadro:\n")
        print(text_region.strip() or "[Vacío / no se reconoció texto]")
        
        # OCR de la página completa
        text_page = pytesseract.image_to_string(state["pil_image"], lang="spa")
        print("\n------------------------------")
        print("📄 Texto completo de la página (primeros 2000 caracteres):\n")
        print((text_page.strip() or "[Vacío / no se reconoció texto]")[:2000])
        print("\n==============================\n")
        
        root.destroy()
    
    # Bind de eventos del mouse
    canvas.bind("<ButtonPress-1>", on_button_press)
    canvas.bind("<B1-Motion>", on_move)
    canvas.bind("<ButtonRelease-1>", on_button_release)
    
    print("Ventana abierta: dibuja un recuadro con el mouse.")
    root.mainloop()

In [8]:
# Ya deberías tener 'pages' del paso abrowse_and_select_region_tk(pages, page_index=page_index)
browse_and_select_region_tk(pages, start_page=0)

Ventana abierta: usa Anterior/Siguiente para cambiar de página y dibuja un recuadro.

📄 Página seleccionada: 1
📌 BBOX (px, original): (1072, 1344, 1292, 1380)

🟦 Texto dentro del recuadro:

T= TASA

------------------------------
📄 Texto completo de la página (primeros 2000 caracteres):

Y Santander

Banco Santander México, S.A.,

O artes MébICO. ESTA DO DE CUENTA

 

MUNICIPIO DE PUERTO VALLARTA JALISCO CODIGO DE CLIENTE NO. 30307689
PERIODO DEL 07-JUN-2024 AL 30-JUN-2024

Z Detalle de movimientos cuenta de cheques.
CUENTA DOLARES MORALES 82-50079894-6

SALDO FINAL DEL PERIODO ANTERIOR: $7,931.50
FECHA FOLIO | DESCRIPCION DEPOSITO RETIRO SALDO
TOTAL 0.00 0.00

 

 

 

 

 

Significado de abreviaturas utilizadas en el estado de cuenta:

ABO= ABONO (5) DEB= DEBITO NO= NUMERO

ANUL=  ANULACION DEP= DEPOSITO NOM= NOMINA

ANT= ANTICIPO DESEM= DESEMPLEO ORD= ORDEN

ANTICIP= ANTICIPADO DEV= DEVOLUCION (ES) P= POR

ASEG= ASEGURAMIENTO DISP= DISPOSICION PAG= PAGARE (5)

AUT= AUTOMATICO DOMIC